In [1]:
%cd ../..

/gpfs/data/schambralab/quantitativeRehabilitation/__lab_member_homes/victor/cvfm4rehab


In [2]:
from postprocess.ia.eval import (
    answers_for_model,
    fm_scores_for_model,
    metrics_for_model,
    metrics_for_models
)

def pivot_fm_scores(df):
    df_long = df.melt(
        id_vars=['patient', 'fm_item'],
        value_vars=['pred_score', 'gt_score'],
        var_name='score_type',
        value_name='score'
    )
    df_long['col'] = df_long['score_type'] + "_" + df_long['patient']
    out = df_long.pivot(index="fm_item", columns="col", values="score")
    return out


def pivot_answers(df):
    melted_df = df.melt(
        id_vars=['patient', 'qid'],
        value_vars=['answer'],
        var_name='answer_type',
    )
    melted_df.drop(columns=['answer_type'], inplace=True)
    melted_df.rename({'value': 'answer'}, axis=1, inplace=True)
    out = (
        melted_df
        .pivot(index="qid", columns="patient", values="answer")
        .add_suffix("_answer")
    )
    return out

In [3]:
from data.utils_strokerehab import DataPaths

In [4]:
MODEL = "qwen2_5_vl_7b"
PATIENT_ORDER = ["C00011", "S0005", "S0001", "S00021"]

In [5]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pivot_table = pivot_answers(answers_for_model(MODEL, tasks=("strokerehab_ia3_3_30",), drop_parsed=False))

In [6]:
qdf = pd.read_csv("./data/ia/fm_item_questions3.csv")
qdf.head()
qdf.drop(columns=["question_type", "sampling", "binary_no_score", "binary_yes_score"], inplace=True)

In [7]:
df = pd.merge(qdf, pivot_table, left_on="qid", right_index=True, how="inner")
df.set_index("qid")[['C00011_answer', 'S0005_answer', 'S0001_answer', 'S00021_answer', 'fm_video', 'question']].to_csv('scrap.csv', index=False)

In [9]:
fm_scores = pivot_fm_scores(fm_scores_for_model(
    MODEL,
    tasks=("strokerehab_ia3_3_30",),
    questions_csv_path=DataPaths.IA_QUESTIONS_PATH3,
    gt_csv_path=DataPaths.IA_SCORES_PATH
))[['gt_score_' + p for p in PATIENT_ORDER] + ['pred_score_' + p for p in PATIENT_ORDER]]
fm_scores

col,gt_score_C00011,gt_score_S0005,gt_score_S0001,gt_score_S00021,pred_score_C00011,pred_score_S0005,pred_score_S0001,pred_score_S00021
fm_item,,,,,,,,
9,2,2,2,0,1,0,0,1
10,2,2,1,0,1,2,1,1
11,2,2,2,0,1,2,1,0


# 12 - 17

In [9]:
fm_scores.sum(axis=0)

col
gt_score_C00011      12
gt_score_S0005       12
gt_score_S0001       11
gt_score_S00021       2
pred_score_C00011     9
pred_score_S0005      8
pred_score_S0001      3
pred_score_S00021     7
dtype: Int64

In [30]:
metrics_for_model(
    MODEL,
    tasks=("strokerehab_ia3_3_30",),
    questions_csv_path=DataPaths.IA_QUESTIONS_PATH3,
    gt_csv_path=DataPaths.IA_SCORES_PATH
)

{'accuracy': 0.37037037037037035, 'apd': 23.25, 'mtsd': 19.25}